# 03 — Same official 40, different speed and acceleration profiles

Compare players mainly within their **Combine position group**, optionally across groups. Select an exact published official 40 time, players, and attempts. Keep both **observed acceleration peaks** and **speed-building intervals**: they describe different aspects of how the recorded run unfolds.

All 418 tracked players and 793 attempts remain available. Single-recording players are included; there is no NFL opportunity filter, no two-attempt requirement for display, and no requirement to match the 10-yard split. The player-level official 40 is not a verified time for each recorded attempt.

Peak time is relative to an observed reference, not confirmed physical launch. Speed bands represent levels of observed speed, not fixed distance splits or verified early/middle/late phases of the 40. Physical units and full sprint boundaries remain unresolved. These profiles describe recorded movement; they do not yet causally explain official times or establish NFL value.

The previous WR-only interval/NFL pilot is preserved in `archive/03_wr_interval_pilot.ipynb`; its existing results and exports are unchanged. This notebook now contains the main all-position analysis.


## 1. Load the all-position preparation handoff

Run Notebook 02 through Section F once. No variables from another notebook are required. The helper module is `scripts/player_comparison.py`. If imports fail, install dependencies with `%pip install pandas numpy scipy matplotlib ipywidgets` in a separate cell and restart the kernel.


In [1]:
from pathlib import Path
import hashlib
import importlib
import json
import sys
import pandas as pd
import numpy as np
from IPython.display import display

COMPARISON_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "scripts" / "player_comparison.py").is_file()),None)

if COMPARISON_ROOT is None:
    raise FileNotFoundError("Open the project folder containing scripts/player_comparison.py")

ALL_PREPARED = COMPARISON_ROOT / "outputs" / "prepared_all_positions"

if not (ALL_PREPARED/"manifest.json").is_file():
    raise FileNotFoundError("Run Notebook 02 Section F to export all position groups first")

all_manifest = json.loads((ALL_PREPARED/"manifest.json").read_text())

if all_manifest.get("schema_version") != 1:
    raise ValueError("Unexpected all-position handoff version")

for name,info in all_manifest["files"].items():
    if hashlib.sha256((ALL_PREPARED/name).read_bytes()).hexdigest()!=info["sha256"]:
        raise ValueError(f"Changed all-position export: {name}; rerun Notebook 02 Section F")

ALL_GROUP_COLS=all_manifest["group_columns"]

all_types={c:"string" for c in ALL_GROUP_COLS}

all_frames=pd.read_csv(ALL_PREPARED/"all_forty_frames.csv",dtype=all_types)
all_frames["time"]=pd.to_datetime(all_frames["time"],errors="raise")

all_roster=pd.read_csv(ALL_PREPARED/"all_player_roster.csv",dtype=all_types)
all_roster["official_forty_hundredths"]=pd.to_numeric(all_roster["official_forty_hundredths"],errors="coerce").astype("Int64")

all_peaks=pd.read_csv(ALL_PREPARED/"all_observed_peaks.csv",dtype=all_types)

if all_roster.duplicated("nfl_id").any():
    raise ValueError("Expected one roster row per player")

if str(COMPARISON_ROOT/"scripts") not in sys.path:
    sys.path.insert(0,str(COMPARISON_ROOT/"scripts"))

import player_comparison
importlib.reload(player_comparison)
print("Players available:",len(all_roster),"Recorded attempts:",len(all_frames[ALL_GROUP_COLS].drop_duplicates()))
display(all_roster.groupby("combine_position").agg(players=("nfl_id","size"),recordings=("recordings","sum")))

PROFILE_OUTPUT = COMPARISON_ROOT / "outputs" / "all_position_profiles"
PROFILE_OUTPUT.mkdir(parents=True,exist_ok=True)

Players available: 418 Recorded attempts: 793


,players,recordings
combine_position,,
DB,102,195
DL,93,177
OL,95,179
TE,34,66
WR,94,176


## 2. Explore selected players: peaks and speed bands together

Choose position → exact official 40 → players → attempts. Cross-position mode is off by default. Use Command-click (macOS) or Ctrl-click (Windows) for multiple selections. Player colors stay consistent across their speed and acceleration curves; line styles distinguish attempts. Peak markers and tables report observed magnitude and time, with boundary flags.

Select speed bands to display elapsed-time measurements for the same attempts. Working bands are **2–4, 4–6, 6–8, and 8–10 raw speed units**, plus the earlier **3–7** interval. These shared thresholds permit consistent comparisons when covered; they are not position-specific performance standards. The coverage table below shows which bands each group can actually support. Never compare durations for different bands as though they measure the same task.

The interactive interval table uses raw supplied speed; the smoothing selector changes peak estimation only. The full analysis below retains raw and 5/7/9-frame interval sensitivity versions. An interval needs bracketed crossings and 0.3 seconds of observed coverage at/above the upper threshold; unsupported intervals remain unavailable, not zero. A player's recording can cover one band but miss another.

Time-zero options are first recorded frame or an observed 3-unit crossing. Neither confirms launch. Intervals themselves are elapsed durations and do not change with the plot's time origin. Multiple draft years remain labeled because recording differences may affect comparisons.

Interpolated crossing times are estimates between 0.1-second samples; displayed decimal places do not establish finer physical timing accuracy.


In [2]:
comparison_app=player_comparison.comparison_widget(all_frames,all_roster,ALL_GROUP_COLS)
display(comparison_app)

## 3. Audit speed-band coverage across all position groups

Apply the same bands to every group before deciding which measurements are suitable. Report availability instead of silently changing thresholds or removing slower players. These are exploratory bands, carried forward before any new NFL outcome tests. They should not be chosen later to maximize a football association.

The first sustained upper crossing and the last bracketed lower crossing before it define the interval. This avoids extrapolation but does not independently establish the sprint phase or remove all pre-sprint movement. Raw versions use observed frames; smoothed versions use only complete centered windows. Differences in availability can reflect method support rather than athlete capability. Any position-specific extension must be named and evaluated separately.


In [3]:
all_intervals=player_comparison.measure_all_intervals(all_frames,ALL_GROUP_COLS)
all_intervals["measured"]=all_intervals["status"].eq("Measured")
interval_coverage=(all_intervals.groupby(["combine_position","speed_band","method"],dropna=False)
    .agg(recordings=("event_id","size"),measured_recordings=("measured","sum"))
    .reset_index())
interval_coverage["available_pct"]=100*interval_coverage["measured_recordings"]/interval_coverage["recordings"]
print("Raw-speed coverage: unsupported intervals remain in the denominator.")
display(interval_coverage.loc[interval_coverage["method"].eq("Raw")].round(2))
print("Raw-speed measurement/failure statuses:")
display(all_intervals.loc[all_intervals["method"].eq("Raw")]
        .groupby(["combine_position","speed_band","status"]).size().rename("recordings").to_frame())
print("The 8-to-10 band is not assumed available for every group or player.")

Raw-speed coverage: unsupported intervals remain in the denominator.


,combine_position,speed_band,method,recordings,measured_recordings,available_pct
3,DB,2-to-4,Raw,195,195,100.00
7,DB,3-to-7,Raw,195,195,100.00
11,DB,4-to-6,Raw,195,195,100.00
15,DB,6-to-8,Raw,195,195,100.00
19,DB,8-to-10,Raw,195,195,100.00
23,DL,2-to-4,Raw,177,177,100.00
27,DL,3-to-7,Raw,177,177,100.00
31,DL,4-to-6,Raw,177,177,100.00
35,DL,6-to-8,Raw,177,177,100.00
39,DL,8-to-10,Raw,177,92,51.98


Raw-speed measurement/failure statuses:


recordings
combine_position speed_band status                                                       
DB               2-to-4     Measured                                                  195
                 3-to-7     Measured                                                  195
                 4-to-6     Measured                                                  195
                 6-to-8     Measured                                                  195
                 8-to-10    Measured                                                  195
DL               2-to-4     Measured                                                  177
                 3-to-7     Measured                                                  177
                 4-to-6     Measured                                                  177
                 6-to-8     Measured                                                  177
                 8-to-10    Measured                                                   92
                            No bracketed upper crossing                                82
                            Upper crossing lacks sustained observed coverage            3
OL               2-to-4     Measured                                                  179
                 3-to-7     Measured                                                  179
                 4-to-6     Measured                                                  179
                 6-to-8     Measured                                                  179
                 8-to-10    Measured                                                    6
                            No bracketed upper crossing                               172
                            Upper crossing lacks sustained observed coverage            1
TE               2-to-4     Measured                                                   65
                            No bracketed lower crossing before upper crossing           1
                 3-to-7     Measured                                                   65
                            No bracketed lower crossing before upper crossing           1
                 4-to-6     Measured                                                   66
                 6-to-8     Measured                                                   66
                 8-to-10    Measured                                                   60
                            No bracketed upper crossing                                 6
WR               2-to-4     Measured                                                  176
                 3-to-7     Measured                                                  176
                 4-to-6     Measured                                                  176
                 6-to-8     Measured                                                  176
                 8-to-10    Measured                                                  176

The 8-to-10 band is not assumed available for every group or player.


## 4. Check repeatability within each position group

Use players with exactly two measured attempts for each feature and report the pair count. Missing intervals never enter as zero. Players with one attempt still remain in the interactive comparison and profile tables. Correlations are calculated within Combine position groups, not pooled across linemen and receivers; pooled correlations could simply reflect differences between groups.

Report both correlations (ranking) and absolute differences (agreement). Draft-year and tracking effects can still affect these summaries. These are exploratory repeatability results, not proof that a feature is an intrinsic athlete trait. Boundary peaks remain flagged, and peak-time repeatability from recording start is not asserted to be launch-time repeatability.


In [4]:
interval_pairs,interval_repeatability=player_comparison.within_position_repeatability(
    all_intervals,"interval_s",["method","speed_band"])
peak_pairs,peak_repeatability=player_comparison.within_position_repeatability(
    all_peaks,"peak_acceleration_raw_per_s",["window_frames"])
print("Raw speed-band repeatability by position:")
display(interval_repeatability.loc[interval_repeatability["method"].eq("Raw")].round(4))
print("Observed peak magnitude repeatability by position — 7-frame estimates:")
display(peak_repeatability.loc[peak_repeatability["window_frames"].eq(7)].round(4))
print("Pair counts vary by observed coverage. One-recording players remain in the comparison controls.")

Raw speed-band repeatability by position:


,combine_position,method,speed_band,paired_players,pearson,spearman,median_absolute_difference,median_absolute_difference_pct,p90_absolute_difference_pct
0,OL,Raw,2-to-4,84,0.4690,0.5286,0.0326,7.4940,23.5231
1,OL,Raw,4-to-6,84,0.7016,0.6424,0.0369,7.5668,18.1353
2,OL,Raw,6-to-8,84,0.8027,0.8387,0.0617,6.7295,19.7722
3,OL,Raw,8-to-10,2,NaN,NaN,0.5555,24.7142,26.2251
4,OL,Raw,3-to-7,84,0.7524,0.7102,0.0796,7.3622,17.1608
20,DL,Raw,2-to-4,84,0.6506,0.6448,0.0264,6.6869,16.3905
21,DL,Raw,4-to-6,84,0.7859,0.8054,0.0239,5.3093,16.3376
22,DL,Raw,6-to-8,84,0.9222,0.8833,0.0467,7.4004,13.9342
23,DL,Raw,8-to-10,39,0.7802,0.7723,0.1366,8.1952,19.8018
24,DL,Raw,3-to-7,84,0.8365,0.8316,0.0490,4.8867,13.5377


Observed peak magnitude repeatability by position — 7-frame estimates:


,combine_position,window_frames,paired_players,pearson,spearman,median_absolute_difference,median_absolute_difference_pct,p90_absolute_difference_pct
1,OL,7,84,0.5810,0.5215,0.3337,7.2513,19.4378
4,DL,7,84,0.6557,0.6930,0.2722,5.3790,15.6761
7,DB,7,93,0.3310,0.3743,0.3250,6.0139,17.3022
10,TE,7,32,0.3038,0.3057,0.3506,6.3955,19.7716
13,WR,7,82,-0.0577,0.2560,0.3798,6.3899,19.9772


Pair counts vary by observed coverage. One-recording players remain in the comparison controls.


## 5. Save all-position profiles and document the scope

Save attempt-level intervals with status, position/method repeatability, player-level interval summaries with attempt counts, and the existing observed-peak measurements. Player means summarize available attempts; they are not a replacement for the individual attempts shown above. Keep official times, Combine and NFL positions, and draft year separate.

This export does not overwrite the earlier WR feature handoff or NFL pilot results. Position-specific NFL linkage is future work; the previous 43-WR speed-interval result remains a limited pilot. Peak timing has not been replaced by an interval metric. Neither metric alone establishes why the official 40 time occurred.


In [5]:
import hashlib
import json
from datetime import datetime,timezone

player_interval_profiles=(all_intervals.loc[all_intervals["status"].eq("Measured")]
    .groupby(["combine_position","nfl_position","draft_year","nfl_id","display_name",
              "official_forty_hundredths","method","speed_band"],dropna=False)
    .agg(mean_interval_s=("interval_s","mean"),measured_attempts=("interval_s","size"),
         min_interval_s=("interval_s","min"),max_interval_s=("interval_s","max"))
    .reset_index())

exports={"all_position_speed_intervals.csv":all_intervals,
         "speed_band_coverage.csv":interval_coverage,
         "speed_band_repeatability_pairs.csv":interval_pairs,
         "speed_band_repeatability.csv":interval_repeatability,
         "peak_repeatability_pairs.csv":peak_pairs,"peak_repeatability.csv":peak_repeatability,
         "player_interval_profiles.csv":player_interval_profiles,
         "observed_peak_profiles.csv":all_peaks,"comparison_roster.csv":all_roster}

profile_manifest={"schema_version":1,"created_utc":datetime.now(timezone.utc).isoformat(),
    "source_all_position_manifest_sha256":hashlib.sha256((ALL_PREPARED/"manifest.json").read_bytes()).hexdigest(),
    "scope":"All Combine position groups; within-group comparison by exact published official time is default",
    "speed_bands_raw":player_comparison.SPEED_BANDS,
    "limitations":["Physical speed units unverified","Launch and full-finish boundaries unverified",
                   "Published 40 is player-level, not mapped to every attempt",
                   "Intervals are speed bands, not distance splits or causal explanations",
                   "Repeatability does not establish NFL value; pilot results are preserved separately"],
    "files":{}}

for name,table in exports.items():
    target=PROFILE_OUTPUT/name;table.to_csv(target,index=False)
    profile_manifest["files"][name]={"rows":len(table),"sha256":hashlib.sha256(target.read_bytes()).hexdigest()}

(PROFILE_OUTPUT/"manifest.json").write_text(json.dumps(profile_manifest,indent=2)+"\n")

print("All-position profiles saved:",PROFILE_OUTPUT)
print("The main comparison retains both observed peaks and broader speed-building intervals.")

All-position profiles saved: /Users/hunterclark/Documents/New project/nfl-big-data-bowl-2027/outputs/all_position_profiles
The main comparison retains both observed peaks and broader speed-building intervals.
